# Implementing an Earley Parser

The file `simple.g` contains a grammar describing arithmetic expressions that only use the operators `+` and `*`.
We will use this grammar as an example.
```
expr: expr '+' product;
expr: product;
    
product: product '*' factor;
product: factor;
       
factor: '(' expr ')';
factor: NUMBER;
```

## Setup and Imports

The file `../lib/parser_generator.ml` provides the function `generate_parser`, which runs the scanner generator
`ocamllex` and the parser generator `Menhir`, and the module `Parse_tree`, which describes parse trees in the style of
`lark`.  (The Python version of this notebook uses the library `lark`.)

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
load_lib "regex.ml";;
open Parse_tree

## Specification of the Grammar

Below are the scanner specification and the grammar that are used to parse our context-free grammar specifications
(like `simple.g`).
* The token `VARIABLE` recognizes a *syntactical variable*.  Syntactical variables start with a lower case letter.
* The token `TERMINAL` recognizes a *token*.  Tokens start with an uppercase variable.
* The token `LITERAL` recognizes a single character that is encclosed in single quotes.
* The scanner discards whitespace and newlines so they do not clutter our syntax tree.
* The rules `rule_list` and `item_list` describe the lists that are written as `rule+` and `item*` in `lark`.
  They return OCaml lists of parse trees.

In [ ]:
let lark_lexer = {|
{
  open Grammar_parser
}

rule token = parse
  | [' ' '\t' '\r' '\012']+       { token lexbuf }
  | '\n'                          { Lexing.new_line lexbuf; token lexbuf }
  | ':'                           { COLON }
  | ';'                           { SEMICOLON }
  | ['a'-'z'] ['a'-'z' '0'-'9' '_']* { VARIABLE (Parse_tree.token "VARIABLE" lexbuf) }
  | ['A'-'Z'] ['A'-'Z' '0'-'9' '_']* { TERMINAL (Parse_tree.token "TERMINAL" lexbuf) }
  | '\'' _ '\''                   { LITERAL (Parse_tree.token "LITERAL" lexbuf) }
  | eof                           { EOF }
  | _                             { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ Lexing.lexeme lexbuf)) }
|}

let lark_grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> VARIABLE TERMINAL LITERAL
%token COLON SEMICOLON EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> rule item
%type  <Parse_tree.tree list> rule_list item_list

%%

start: l = rule_list EOF                          { Tree ("start", l) }

rule_list: r = rule                               { [r] }
         | r = rule l = rule_list                 { r :: l }

rule: v = VARIABLE COLON l = item_list SEMICOLON  { Tree ("rule", Token v :: l) }

item_list: /* empty */                            { [] }
         | i = item l = item_list                 { i :: l }

item: v = VARIABLE                                { Token v }
    | t = TERMINAL                                { Token t }
    | l = LITERAL                                 { Token l }
|}

We generate the scanner and an *LALR* parser.  We do not use a Transformer here.

In [ ]:
generate_parser ~name:"grammar" ~lexer:lark_lexer ~grammar:lark_grammar

## Generating the Abstract Syntax Tree (AST)

We walk the parse tree generated by the parser and convert it into a nested list.

Our Earley parser expects the grammar to be a list of rules, where each rule is a list of strings (e.g., `["expr"; "expr"; "+"; "product"]`). We extract these strings recursively, taking care to strip the surrounding quote marks from `LITERAL` tokens.

In Python, the function `ast_to_rules` returns a string for a token, a list of strings for a rule, and a list of
lists for the whole grammar.  In OCaml, a function has a single result type.  Therefore, the tokens are handled by the
local function `symbol` and the rules by the local function `rule`.

In [ ]:
let ast_to_rules (node : tree) : string list list =
  let symbol (item : tree) : string =
    match item with
    (* Strip the surrounding single quotes (e.g., "'+'" -> "+") *)
    | Token t when t.kind = "LITERAL" -> String.sub t.value 1 (String.length t.value - 2)
    | Token t -> t.value
    | Tree _ -> invalid_arg "ast_to_rules: unexpected node"
  in
  let rule (r : tree) : string list =
    match r with
    | Tree ("rule", items) -> List.map symbol items
    | _ -> invalid_arg "ast_to_rules: rule expected"
  in
  match node with
  | Tree ("start", rules) -> List.map rule rules
  | _ -> invalid_arg "ast_to_rules: start expected"

The `parse` function reads the grammar file, parses it into a parse tree, and converts it into our nested list structure.

In [ ]:
let parse (file : string) : string list list =
  let grammar_str = read_file file in
  print_endline grammar_str;
  let raw_tree = parse_string Grammar_parser.start Grammar_lexer.token grammar_str in
  ast_to_rules raw_tree

In [ ]:
parse "simple.g"

## Earley's Algorithm

Given a context-free grammar $G = \langle V, \Sigma, R, s \rangle$ and a string $w = x_1x_2 \cdots x_n \in \Sigma^*$ of length $n$,
an *Earley item* is a pair of the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
such that
- $(a \rightarrow \alpha \beta) \in R\quad$  and
- $k \in \{0,1,\cdots,n\}$.

In Python, the class `EarleyItem` represents a single *Earley item*.  In OCaml, we define the module `EarleyItem`,
which contains the record type `t` with the following fields:
- `mVariable` is the variable $a$,
- `mAlpha` is $\alpha$,
- `mBeta` is $\beta$, and
- `mIndex` is $k$.

The function `make` creates an Earley item.  It corresponds to the constructor of the Python class.

Since we later have to store Earley items in sets, the Python class has to implement the methods `__eq__`, `__ne__`,
and `__hash__`.  In OCaml, this is not necessary: records are compared *structurally*, i.e. two records are equal iff
their fields are equal, so they can be stored in sets of type `FSet.t` immediately.  The function `to_string` corresponds
to the method `__repr__`.  It is installed as the printer of Earley items.

In [ ]:
module EarleyItem = struct
  type t = { mVariable : string; mAlpha : string list; mBeta : string list; mIndex : int }

  let make (variable : string) (alpha : string list) (beta : string list) (index : int) : t =
    { mVariable = variable; mAlpha = alpha; mBeta = beta; mIndex = index }

  let to_string (self : t) : string =
    let alphaStr = String.concat " " self.mAlpha in
    let betaStr  = String.concat " " self.mBeta in
    Printf.sprintf "<%s → %s • %s, %d>" self.mVariable alphaStr betaStr self.mIndex

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer EarleyItem.pp

Given an Earley item `self`, the function `isComplete` checks, whether the Earley item `self` has the form
$$\langle A \rightarrow \alpha \bullet, k \rangle,$$
i.e. whether the $\bullet$ is at the end of the grammar rule.

The Python version adds this function to the class `EarleyItem` via the assignment `EarleyItem.isComplete = isComplete`.
In OCaml, we *extend* the module `EarleyItem`: the new module `EarleyItem` *includes* everything from the old module
and adds the function `isComplete`.  Afterwards, the function is called as `EarleyItem.isComplete item`.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let isComplete (self : t) : bool = self.mBeta = []
end

The function `sameVar self c` checks, whether the item following the dot is the same as the variable
given as argument, i.e. `sameVar self c` returns `true` if `self` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet c\beta, k \rangle.$$

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let sameVar (self : t) (c : string) : bool =
    match self.mBeta with
    | b :: _ -> b = c
    | [] -> false
end

The function `scan self t` checks, whether the item following the dot matches the token `t`,
i.e. `scan self t` returns `true` if `self` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet T\beta, k \rangle.$$
The argument $T$ can either be the name of a token or a literal.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let scan (self : t) (t : string) : bool =
    match self.mBeta with
    | b :: _ -> b = t
    | [] -> false
end

Given an Earley item, this function returns the name of the variable following the dot.  If there is no variable
following the dot, the function returns `None`.  Its result has the type `string option`: it is either `None` or
`Some var`.  The function can distinguish variables from token names because variable names consist only of lower
case letters.  The function `is_lower` corresponds to Python's method `islower`: it checks that the string contains a
lower case letter, but no upper case letter.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let is_lower (s : string) : bool =
    String.exists (fun c -> 'a' <= c && c <= 'z') s && not (String.exists (fun c -> 'A' <= c && c <= 'Z') s)

  let nextVar (self : t) : string option =
    match self.mBeta with
    | var :: _ when var.[0] <> '\'' && is_lower var -> Some var
    | _ -> None
end

The function `moveDot self` moves the $\bullet$ in the Earley item `self`, where `self` has the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
over the next variable, token, or literal in $\beta$.  It assumes that $\beta$ is not empty.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let moveDot (self : t) : t =
    match self.mBeta with
    | b :: rest -> make self.mVariable (self.mAlpha @ [ b ]) rest self.mIndex
    | [] -> invalid_arg "moveDot: the dot is already at the end"
end

The module `Grammar` represents a context free grammar.  It stores a list of the rules of the grammar.
Each grammar rule of the form
$$ a \rightarrow \beta $$
is stored as the list $[a] \mathbin{@} \beta$.  The start symbol is assumed to be the variable on the left hand side of
the first rule. To distinguish syntactical variables from tokens, variables contain only lower case letters,
while tokens either contain only upper case letters or they start and end with a single quote character "`'`".

The functions `startItem` and `finishItem` call the function `startVar`, which is only defined later.  Therefore, the
module contains a *forward reference* `startVar_ref`, i.e. a mutable box that will later contain the function
`startVar`.  `!startVar_ref` is the function that is currently stored in the box.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

module Grammar = struct
  type t = { mRules : string list list }

  let make (rules : string list list) : t = { mRules = rules }

  let startVar_ref : (t -> string) ref = ref not_yet_defined
end

The function `startItem` returns the Earley item
$$ \langle\hat{s} \rightarrow \bullet s, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is the new variable `Start`.

In [ ]:
module Grammar = struct
  include Grammar

  let startItem (self : t) : EarleyItem.t = EarleyItem.make "Start" [] [ !startVar_ref self ] 0
end

The function `finishItem` returns the Earley item
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is a new variable.

In [ ]:
module Grammar = struct
  include Grammar

  let finishItem (self : t) : EarleyItem.t = EarleyItem.make "Start" [ !startVar_ref self ] [] 0
end

The function `startVar` returns the start variable of the grammar.  It is assumed that
the first rule grammar starts with the start variable of the grammar.

In [ ]:
module Grammar = struct
  include Grammar

  let startVar (self : t) : string = List.hd (List.hd self.mRules)
end
;;
Grammar.startVar_ref := Grammar.startVar

The module `EarleyParser` implements the [parsing algorithm of Jay Earley](https://en.wikipedia.org/wiki/Earley_parser).
Its record type `t` has the following fields:
- `mGrammar` is the grammar that is used to parse the given token string.
- `mString` is the array of tokens and literals that has to be parsed.

   As a hack, the first element of this array is the empty string.
   Therefore, `mString.(i)` is the $i^\textrm{th}$ token.
- `mStateList` is an array of sets of *Earley items*.  If $n$ is the length of the given token string
  (excluding the first element), then $Q_i = \texttt{mStateList.(i)}$.
  The idea is that the set $Q_i$ is the set of those *Earley items* that the parser could be in
  when it has read the tokens `mString.(1)`, $\cdots$,  `mString.(i)`.  $Q_0$ is initialized as follows:
  $$ Q_0 = \bigl\{\langle\hat{s} \rightarrow \bullet s, 0\rangle\bigr\}. $$
  Since the sets are changed by the algorithm, `mStateList` is an *array*, whose elements can be replaced.

The *Earley items* are interpreted as follows: If we have
$$ \langle c \rightarrow \alpha \bullet \beta, k\rangle \in Q_i, $$
then we know the following:
- After having read the tokens `mString[:k+1]` the parser tries to parse the variable $c$
  in the token string `mString[k+1:]`.
- After having read the token string `mString[k+1:i+1]` the parser has already recognized $\alpha$
  and now needs to recognize $\beta$ in the token string `mString[i+1:]` in order to parse the variable $c$.

The function `make` creates a parser.  The method `parse` calls the methods `complete`, `predict`, and `scan`, which
are defined later, so the module contains forward references for them.

In [ ]:
module EarleyParser = struct
  type t = { mGrammar : Grammar.t; mString : string array; mStateList : EarleyItem.t FSet.t array }

  let make (grammar : Grammar.t) (tokenList : string list) : t =
    let mString    = Array.of_list ("" :: tokenList) in   (* hack so mString.(1) is the first token *)
    let mStateList = Array.make (List.length tokenList + 1) FSet.empty in
    mStateList.(0) <- FSet.singleton (Grammar.startItem grammar);
    { mGrammar = grammar; mString; mStateList }

  let complete_ref : (t -> int -> bool) ref = ref not_yet_defined
  let predict_ref  : (t -> int -> bool) ref = ref not_yet_defined
  let scan_ref     : (t -> int -> unit) ref = ref not_yet_defined
end

The function `parse` implements Earley's algorithm.  For all states
$Q_1$, $\cdots$, $Q_n$ we proceed as follows:
- We apply the *completion* operation followed by the *prediction* operation.
  This is done until no more states are added to $Q_i$.

  (The inner `while` loop is not necessary if the grammar does not contain $\varepsilon$-rules.)
- Finally, the *scanning* operation is applied to $Q_i$.  This operation adds
  items to the set $Q_{i+1}$.

After $Q_i$ has been computed, we proceed to process $Q_{i+1}$.
Parsing is successful iff
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n $$

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let parse (self : t) : unit =
    (* run Earley's algorithm *)
    let n = Array.length self.mString - 1 in   (* mString.(0) = "" *)
    for i = 0 to n do
      let rec loop () =
        let change = !complete_ref self i in
        let change = !predict_ref self i || change in
        if change then loop ()
      in
      loop ();
      !scan_ref self i
    done;
    if FSet.mem (Grammar.finishItem self.mGrammar) self.mStateList.(n) then print_endline "Parsing successful!"
    else print_endline "Parsing failed!"
end

The method `complete self i` applies the completion operation to the state $Q_i$:
If we have
 - $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$  and
 - $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$,
then the parser tried to parse the variable $c$ after having read `mString[:j+1]`
and we know that 
$$ c \Rightarrow^* \texttt{mString[j+1:i+1]}, $$
i.e. the parser has recognized $c$ after having read `mString[j+1:i+1]`.
Therefore the parser should proceed to recognize $\delta$ in state $Q_i$.
Therefore we add the *Earley item* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ to the set $Q_i$:
$$\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i \wedge
  \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j \;\rightarrow\;
          Q_i := Q_i \cup \bigl\{ \langle a \rightarrow \beta c \bullet \delta, k\rangle \bigr\}
$$


In [ ]:
module EarleyParser = struct
  include EarleyParser

  let complete (self : t) (i : int) : bool =
    let change = ref false in
    let rec loop () =
      let qi = self.mStateList.(i) in
      let newQi =
        let open FSet in
        let* item = qi in
        if EarleyItem.isComplete item then
          let c  = item.EarleyItem.mVariable in
          let qj = self.mStateList.(item.EarleyItem.mIndex) in
          let* newItem = qj in
          if EarleyItem.sameVar newItem c then return (EarleyItem.moveDot newItem) else empty
        else empty
      in
      if not (FSet.subset newQi qi) then begin
        change := true;
        self.mStateList.(i) <- FSet.union qi newQi;
        loop ()
      end
    in
    loop ();
    !change
end
;;
EarleyParser.complete_ref := EarleyParser.complete

The method `predict self i` applies the prediction operation to the state $Q_i$: 
If $\langle a \rightarrow \beta \bullet c \delta, k \rangle \in Q_i$, then
the parser tries to recognize $c\delta$ after having read `mString[:i+1]`.  To this end
it has to parse $c$ in the string `mString[i+1:]`.
Therefore, if $c \rightarrow \gamma$ is a rule of our grammar,
we add the *Earley item* $\langle c \rightarrow \bullet \gamma, i\rangle$ to the set $Q_i$:
$$ \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_i 
       \wedge (c \rightarrow \gamma) \in R 
       \;\rightarrow\;
       Q_i := Q_i \cup\bigl\{ \langle c \rightarrow \bullet\gamma, i\rangle\bigr\}.
$$
As the right hand side $\gamma$ might start with a variable, the function uses a fix point iteration
until no more *Earley items* are added to $Q_i$.

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let predict (self : t) (i : int) : bool =
    let change = ref false in
    let rec loop () =
      let qi = self.mStateList.(i) in
      let newQi =
        let open FSet in
        let* item = qi in
        match EarleyItem.nextVar item with
        | Some c ->
            FSet.of_list
              (List.filter_map
                 (fun rule -> if c = List.hd rule then Some (EarleyItem.make c [] (List.tl rule) i) else None)
                 self.mGrammar.Grammar.mRules)
        | None -> empty
      in
      if not (FSet.subset newQi qi) then begin
        change := true;
        self.mStateList.(i) <- FSet.union qi newQi;
        loop ()
      end
    in
    loop ();
    !change
end
;;
EarleyParser.predict_ref := EarleyParser.predict

The function `scan self i` applies the scanning operation to the state $Q_i$.

If $\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i$ and $T$ is a token,
then the parser tries to recognize the right hand side of the grammar rule
$$ a \rightarrow \beta T \gamma$$
and after having read `mString[k+1:i+1]` it has already recognized  $\beta$.
If we now have `mString.(i+1) = T`, then the parser still has to recognize $\gamma$ in `mString[i+2:]`.
Therefore, the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle$ is added to
the set $Q_{i+1}$:
$$\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i \wedge x_{i+1} = T
       \;\rightarrow\;
       Q_{i+1} := Q_{i+1} \cup \bigl\{ \langle a \rightarrow \beta T \bullet \gamma, k\rangle \bigr\}
$$

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let scan (self : t) (i : int) : unit =
    let qi = self.mStateList.(i) in
    let n  = Array.length self.mString - 1 in   (* remember mString.(0) = "" *)
    if i + 1 <= n then begin
      let a = self.mString.(i + 1) in
      FSet.iter
        (fun item ->
          if EarleyItem.scan item a then
            self.mStateList.(i + 1) <- FSet.add (EarleyItem.moveDot item) self.mStateList.(i + 1))
        qi
    end
end
;;
EarleyParser.scan_ref := EarleyParser.scan

In [ ]:
(* the module Re has been loaded at the beginning of the notebook *)

The function `tokenize` transforms the string `s` that is to be parsed into a list of tokens. See below for an example.
The regular expression contains five groups.  Therefore, we use the function `Re.findall_groups`, which returns the list
of the strings matched by the groups for every match.  Exactly one of these strings is not empty.

In [ ]:
let tokenize (s : string) : string list =
  (* Transform the string s into a list of tokens.  The string s
     is supposed to represent an arithmetic expression. *)
  let lexSpec = {|([ \t]+)        |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([()])          |  # parentheses
                  ([-+*/])        |  # arithmetical operators
                  (.)                # unrecognized character
               |}
  in
  let tokenList = Re.findall_groups ~flags:Re.[VERBOSE] lexSpec s in
  List.filter_map
    (function
      | [ ws; number; parenthesis; operator; error ] ->
          if ws <> "" then None                       (* skip blanks and tabs *)
          else if number <> "" then Some "NUMBER"
          else if parenthesis <> "" then Some parenthesis
          else if operator <> "" then Some operator
          else Some (Printf.sprintf "ERROR(%s)" error)
      | _ -> assert false)
    tokenList

In [ ]:
tokenize "1 + 2 * 3"

The function `test` takes two arguments.
- `file` is the name of a file containing a grammar,
- `word` is a string that should be parsed.

`word` is first tokenized. Then the grammar is parsed into a parse tree, converted into a list of rules using our `ast_to_rules` function, and passed directly to the function `Grammar.make`.  The token list is then parsed using *Earley's algorithm*.

In [ ]:
let test (file : string) (word : string) : unit =
  let grammarStr = read_file file in
  print_endline grammarStr;
  (* generate the rules using the parser and convert them using ast_to_rules *)
  let raw_tree  = parse_string Grammar_parser.start Grammar_lexer.token grammarStr in
  let rules     = ast_to_rules raw_tree in
  let grammar   = Grammar.make rules in
  let tokenList = tokenize word in
  let ep        = EarleyParser.make grammar tokenList in
  EarleyParser.parse ep

In [ ]:
test "simple.g" "1 + 2 * 3"